In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.2 Bytes and byte-level BPE

Byte-level BPE starts from the 256 raw bytes (so *any* text round-trips, no
OOV) and merges frequent adjacent pairs into new tokens. Fewer tokens, same
text.

In [ ]:
from pocketlm import BPETokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
bpe = BPETokenizer().train(corpus[:20000], 400)
text = "to be or not to be, that is the question"
ids = bpe.encode(text)
print("raw bytes:", len(text.encode("utf-8")), " bpe tokens:", len(ids))
print("round trip ok:", bpe.decode(ids) == text)

Byte-level means even an emoji round-trips, it is just several byte tokens.
Merges never lengthen the sequence, so BPE is a strict win on length.

In [ ]:
assert bpe.decode(ids) == text
assert len(ids) <= len(text.encode("utf-8"))
assert bpe.decode(bpe.encode("\U0001f600")) == "\U0001f600"